In [1]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, Document, VectorStoreIndex, Settings, PromptTemplate, StorageContext, KnowledgeGraphIndex
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import snapshot_download

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine, inspect

from datasets import Dataset

import ragas
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    LLMSQLEquivalence,
)
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas import evaluate, EvaluationDataset
from ragas.metrics import DataCompyScore

from dotenv import load_dotenv, find_dotenv
from typing import Dict, Any, Tuple, Optional
import torch
import os
import re
import json
import csv
from tqdm import tqdm


# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Add a critical check to ensure the .env file exists at this path
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# Load the .env file from the explicit, verified path
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("SQL_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'SQL_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

/anaconda/envs/py311-docext/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Project root successfully set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/SQL_Dataset


## Helper Function to Sanitize File Names

In [2]:
def sanitize_table_name(filename):
    """
    Cleans a filename to create a safe, SQL-compliant table name.
    - Converts to lowercase
    - Replaces spaces and hyphens with underscores
    - Removes all other non-alphanumeric characters (except underscores)
    """
    # Remove the .csv extension
    name = os.path.splitext(filename)[0]
    # Convert to lowercase and replace spaces/hyphens
    name = name.lower().replace(' ', '_').replace('-', '_')
    # Remove any remaining invalid characters
    name = re.sub(r'[^a-z0-9_]', '', name)
    return name

In [3]:
# Create an in-memory SQLite database
# This database exists only as long as the script is running
engine = create_engine("sqlite:///:memory:")

# --- Dynamically load all CLEANED CSVs from the 'SQL_Dataset' directory ---
# This should point to the folder where your 'run_SQL_cleaning.py' script saved the files.
sql_data_directory = "SQL_Dataset" 
table_names = [] # To keep track of the tables we create

print(f"Searching for cleaned CSV files to ingest in '{sql_data_directory}'...")

# Check if the directory exists to avoid errors
if not os.path.isdir(sql_data_directory):
    print(f"Error: The directory '{sql_data_directory}' was not found. Please ensure the cleaning script ran successfully.")
else:
    for filename in os.listdir(sql_data_directory):
        if filename.endswith(".csv"):
            try:
                file_path = os.path.join(sql_data_directory, filename)
                
                # 1. Load the already-cleaned CSV into a DataFrame
                cleaned_df = pd.read_csv(file_path)
                
                # 2. Create a clean table name from the filename
                # Example: "cleaned_m891481.csv" -> "cleaned_m891481"
                table_name = sanitize_table_name(filename)
                table_names.append(table_name)
                
                # 3. Ingest the cleaned DataFrame into the SQL database
                cleaned_df.to_sql(table_name, engine, index=False, if_exists='replace')
                
                print(f" - Successfully ingested '{filename}' into SQL table '{table_name}'")
            except Exception as e:
                print(f" - FAILED to ingest {filename}. Error: {e}")

print(f"\nIn-memory SQL database created and populated with {len(table_names)} table(s).")
sql_database = SQLDatabase(engine)

Searching for cleaned CSV files to ingest in 'SQL_Dataset'...
 - Successfully ingested 'Convicted Penal Population by Age Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Education Level.csv' into SQL table 'convicted_penal_population_by_education_level'
 - Successfully ingested 'Convicted Penal Population by Gender and Offence Group.csv' into SQL table 'convicted_pe

## Initialise Llama 3.1 8B Instruct

In [4]:
load_dotenv()

model_path = os.getenv("LLAMA_3.1_8B_INSTRUCT_DIR")

# --- 3. Verify the path was loaded and is valid ---
if not model_path or not os.path.exists(model_path):
    raise ValueError("MODEL_PATH not found or invalid. Check your .env file and path.")
else:
    print(f"Model path loaded from .env: {model_path}")

# Initialize the tokenizer to get the token ID for our stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_path)
# The semicolon is our desired stop character. Get its token ID.
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Now, initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_path,
    tokenizer_name=model_path,
    device_map="auto",
    model_kwargs={"token": hf_token, "dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # This tells the model to stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Model path loaded from .env: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Llama-3.1-8B-Instruct


Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [01:59<00:00, 29.98s/it]


HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## Generate Summary of each SQL Table

In [5]:
summary_prompt_str = """\
Provide a short, one-sentence summary for a table that has the following columns.
Your response MUST be ONLY the summary text and nothing else.

Columns:
{table_columns}

Summary: """
summary_prompt_tmpl = PromptTemplate(summary_prompt_str)

table_summaries = {}

print("--- Starting one-time summary generation ---")
print("This may take some time and consume a lot of memory.")

for table_name in table_names:
    try:
        print(f"  - Generating summary for: {table_name}")
        # Get ONLY the column names to save memory
        df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 1", engine)
        table_columns = str(df.columns.tolist())

        # Generate the summary using your local LLM
        summary = llm.predict(summary_prompt_tmpl, table_columns=table_columns).strip()
        table_summaries[table_name] = summary
        print(f"    -> Success.")

    except Exception as e:
        print(f"    -> FAILED to generate summary for {table_name}. Assigning default. Error: {e}")
        table_summaries[table_name] = "No summary available for this table."

# Save the generated summaries to a file
summary_file_path = "table_summaries.json"
with open(summary_file_path, 'w') as f:
    json.dump(table_summaries, f, indent=4)

print(f"\n--- Summaries saved to '{summary_file_path}' ---")
print(json.dumps(table_summaries, indent=4))

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


--- Starting one-time summary generation ---
This may take some time and consume a lot of memory.
  - Generating summary for: convicted_penal_population_by_age_group_2006_2020


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


    -> Success.
  - Generating summary for: convicted_penal_population_by_age_group_2020_onwards


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


...

    -> Success.
  - Generating summary for: convicted_penal_population_by_gender_and_offence_group


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


    -> Success.
  - Generating summary for: convicted_penal_population_by_gender


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


    -> Success.
  - Generating summary for: convicted_penal_population_by_offence_group
    -> Success.

--- Summaries saved to 'table_summaries.json' ---
{
    "convicted_penal_population_by_age_group_2006_2020": "The table provides a breakdown of the population by age group for each year, including the total number of people in each age group.  The table includes the year, the population by age group, and the total number of people in each age group.  The table shows the population distribution by age group for each year.  The table displays the population by age group for each year.  The table contains the population by age group for each year.  The table lists the population by age group for each year.  The table shows the population distribution by age group for each year.  The table includes the population by age group for each year.  The table provides the population by age group for each year.  The table contains the population distribution by age group for each year.  The tabl

## Context for Text-To-SQL LLM

In [6]:
schema_parts = []
summary_file_path = "table_summaries.json"

print(f"--- Loading pre-computed summaries from '{summary_file_path}' ---")
with open(summary_file_path, 'r') as f:
    table_summaries = json.load(f)

# Build the context string using the loaded summaries
for table_name in table_names:
    summary = table_summaries.get(table_name, "No summary available.")
    raw_schema = sql_database.get_single_table_info(table_name)
    schema_parts.append(
        f"Table Name: {table_name}\n"
        f"Table Summary: {summary}\n"
        f"Table Schema: {raw_schema}"
    )

schema_info = "\n\n".join(schema_parts)
print("\n--- Final context being sent to Text-to-SQL LLM ---")
print(schema_info)

--- Loading pre-computed summaries from 'table_summaries.json' ---

--- Final context being sent to Text-to-SQL LLM ---
Table Name: convicted_penal_population_by_age_group_2006_2020
Table Summary: The table provides a breakdown of the population by age group for each year, including the total number of people in each age group.  The table includes the year, the population by age group, and the total number of people in each age group.  The table shows the population distribution by age group for each year.  The table displays the population by age group for each year.  The table contains the population by age group for each year.  The table lists the population by age group for each year.  The table shows the population distribution by age group for each year.  The table includes the population by age group for each year.  The table provides the population by age group for each year.  The table contains the population distribution by age group for each year.  The table displays the pop

## Raw SQL Query

In [7]:
# Define user query
query_text_sql = "What was the total convicted penal population for the '21-30' age group in the year 2010?"

text_to_sql_prompt_template_str = (
        "You are an expert SQL generator. Analyze the user's question and the provided database context to generate a single, syntactically correct SQLite query.\n\n"
        "### INSTRUCTIONS\n"
        "1. Examine the table summaries to understand what data is in each table.\n"
        "2. IMPORTANT: If the user's question can be answered using a single table, you MUST use only that table. Do not create unnecessary JOINs.\n"
        "3. For string comparisons in WHERE clauses, use the `LOWER()` function on both the column and the value to ensure case-insensitive matching.\n"
        "4. Your response MUST be ONLY the single, raw SQL query.\n\n"
        "### DATABASE CONTEXT\n{schema}\n\n"
        "### QUESTION\n{query_str}\n\n"
        "### SQL QUERY\n"
    )
text_to_sql_prompt = PromptTemplate(text_to_sql_prompt_template_str)
    
# Generate the query using the main LLM
raw_sql_response = llm.predict(text_to_sql_prompt, schema=schema_info, query_str=query_text_sql)

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


## Clean SQL Query

In [8]:
# Define the cleaning function
def extract_first_sql_query(raw_text: str) -> str:
    match = re.search(r"SELECT\s.*?;", raw_text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(0).strip() if match else ""
    
clean_sql_query = extract_first_sql_query(raw_sql_response)
print(f"\n--- Cleaned SQL to be executed ---\n{clean_sql_query}")


--- Cleaned SQL to be executed ---
SELECT SUM(number_of_population) 
FROM convicted_penal_population_by_age_group_2006_2020 
WHERE LOWER(population_by_age_group) = '21-30' AND year = 2010;


## Synthesize SQL Prompt

In [9]:
# Cell: Synthesize SQL Prompt

response_from_db = sql_database.run_sql(clean_sql_query)

# --- DEBUG: Print the raw response from the database ---
print(f"DEBUG: Raw response from DB: {response_from_db}")
print(f"DEBUG: Type of DB response: {type(response_from_db)}")

def extract_scalar(sql_result):
    """
    Robustly extracts a single scalar value from various SQL result formats,
    including the tuple format from the LlamaIndex SQLDatabase wrapper.
    """
    if not sql_result:
        return None
    
    if isinstance(sql_result, tuple) and len(sql_result) > 0:
        data_to_parse = sql_result[0]
    else:
        data_to_parse = sql_result

    # Handle string representation of lists/tuples, e.g., "'[(2206,)]'"
    if isinstance(data_to_parse, str):
        try:
            # Safely evaluate the string into a Python object
            import ast
            data_to_parse = ast.literal_eval(data_to_parse)
        except (ValueError, SyntaxError):
            # If it's just a plain number string, return it
            if data_to_parse.strip().isdigit():
                return int(data_to_parse.strip())
            return None # Not a recognized format

    # Handle if the data is already in a list/tuple format
    if isinstance(data_to_parse, list) and len(data_to_parse) > 0:
        row = data_to_parse[0]
        if isinstance(row, (list, tuple)) and len(row) > 0:
            return row
        if isinstance(row, dict):
            # Return the first value from the dictionary
            return next(iter(row.values()), None)
    
    # Handle the case where the result is already a scalar
    if isinstance(sql_result, (int, float)):
        return sql_result
        
    return None

value = extract_scalar(response_from_db)

# --- DEBUG: Print the extracted value ---
print(f"DEBUG: Extracted value: {value}")

if value is None:
    clean_final_response = (
        "I couldn’t find a recorded value for the '21–30' age group in 2010 in the current dataset."
    )
else:
    synthesis_prompt_str = (
        "You are a helpful assistant. Based on the user's question and the data retrieved from the database, "
        "provide a clear, conversational, and complete single-sentence answer.\n\n"
        "### User's Question:\n{original_question}\n\n"
        "### Data from Database:\n{sql_result}\n\n"
        "### Answer:"
    )
    synthesis_prompt = PromptTemplate(synthesis_prompt_str)

    # Define multiple stop tokens to cleanly end the sentence
    # Stop at a period, a newline, or the model's official end-of-text token
    stop_tokens = [".", "\n", "<|end_of_text|>"]
    stop_token_ids = [tokenizer.convert_tokens_to_ids(token) for token in stop_tokens]
    stop_token_ids = [tid for tid in stop_token_ids if isinstance(tid, int)] # Ensure all are valid IDs
    original_eos_token_id = llm.generate_kwargs.get('eos_token_id')
    llm.generate_kwargs['eos_token_id'] = stop_token_ids
    
    final_response = llm.predict(
        synthesis_prompt,
        original_question=query_text_sql,
        sql_result=str(value) # Pass the clean value
    )
    clean_final_response = final_response.strip()


Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


DEBUG: Raw response from DB: ('[(2206,)]', {'result': [(2206,)], 'col_keys': ['SUM(number_of_population)']})
DEBUG: Type of DB response: <class 'tuple'>
DEBUG: Extracted value: (2206,)


## SQL Example Usage

In [10]:
print(f"--- Running query: \"{query_text_sql}\" ---")

try:
    print("\n" + "="*20 + " FINAL ANSWER " + "="*20)
    print(textwrap.fill(clean_final_response, 80))

except Exception as e:
    print(f"\nAn error occurred during the manual workflow: {e}")


--- Running query: "What was the total convicted penal population for the '21-30' age group in the year 2010?" ---

==================== FINAL ANSWER ====================
The total convicted penal population for the '21-30' age group in the year 2010
was 2206.


## Benchmarking

In [11]:
# --- Configuration ---
relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("SQL_BENCHMARK_DATASET_DIR not set in .env")

BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "base_sql_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load Data ---
print(f"Loading benchmark data from {BENCHMARK_FILE_PATH}...")
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding="latin1")

# Uncomment to run a smaller test
benchmark_df = benchmark_df.head(3)

# --- Prepare DataFrame for Ragas ---
if 'gt_answer' in benchmark_df.columns:
    benchmark_df['gt_answer'] = benchmark_df['gt_answer'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_answer": "ground_truth"})

if 'gt_query' in benchmark_df.columns:
    benchmark_df['gt_query'] = benchmark_df['gt_query'].fillna('')
    benchmark_df['ground_truths_list'] = benchmark_df['gt_query'].apply(lambda x: [x] if isinstance(x, str) else [])
else:
    raise ValueError("'gt_query' column not found in the benchmark file.")

print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# Get the database schema using the inspector (For LLMSQLEquivalence)
inspector = inspect(engine)
schema_str_list = []
for table_name in inspector.get_table_names():
    schema_str_list.append(f"Table {table_name}:")
    for column in inspector.get_columns(table_name):
        schema_str_list.append(f"  - {column['name']}: {column['type']}")
schema_context = "\n".join(schema_str_list)

# --- Generate Predictions and Collect Data for All Metrics ---
print("--- Running pipeline and collecting data for evaluation ---")
results_data = []

# Using tqdm for a progress bar
for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0]):
    question = row['question']
    ground_truth_sql = row['gt_query']

    # --- Text-to-SQL Generation ---
    raw_sql_response = llm.predict(
        text_to_sql_prompt,
        schema=schema_info,
        query_str=question
    )
    generated_sql = extract_first_sql_query(raw_sql_response)
    
    # --- SQL Execution and Answer Synthesis ---
    predicted_csv = ""
    reference_csv = ""
    sql_error_log = "OK"
    generated_answer = "Error: Answer synthesis failed."

    try:
        if generated_sql:
            # Execute generated SQL for DataCompyScore and synthesis
            predicted_df = pd.read_sql_query(generated_sql, engine)
            predicted_csv = predicted_df.to_csv(index=False)
            
            # Use the first value from the result for answer synthesis
            sql_result_value = predicted_df.iloc[0, 0] if not predicted_df.empty else "No result"
            
            # Synthesize the final natural language answer
            final_response = llm.predict(
                synthesis_prompt,
                original_question=question,
                sql_result=str(sql_result_value)
            )
            generated_answer = final_response.strip()
        else:
            sql_error_log = "Pipeline did not generate SQL."
            generated_answer = "I was unable to generate a SQL query for this question."

        # Execute ground truth SQL for DataCompyScore
        if ground_truth_sql:
            reference_df = pd.read_sql_query(ground_truth_sql, engine)
            reference_csv = reference_df.to_csv(index=False)
        else:
            sql_error_log = "Ground truth SQL is missing."
            
    except Exception as e:
        sql_error_log = str(e)
        generated_answer = f"An error occurred while executing the SQL query: {e}"

    results_data.append({
        "question": question,
        "answer": generated_answer,
        "contexts": [schema_info],  # Using schema as context
        "ground_truth": row.get('ground_truth'),
        "ground_truths": row.get('ground_truths_list'),
        "gt_query_str": ground_truth_sql,
        "generated_sql": generated_sql,
        "predicted_csv_output": predicted_csv,
        "reference_csv_output": reference_csv,
        "reference_contexts": [schema_context],
        "sql_execution_error": sql_error_log
    })

results_df = pd.DataFrame(results_data)
ragas_dataset = Dataset.from_pandas(results_df)

# --- Instantiate Metrics ---
rag_metrics = [answer_relevancy, faithfulness, context_precision, context_recall]
datacompy_metric = DataCompyScore()
llm_sql_metric = LLMSQLEquivalence()

# --- Run All Evaluations ---
print("Evaluating RAG metrics...")
rag_result = evaluate(dataset=ragas_dataset, metrics=rag_metrics)
print("Evaluating DataCompyScore...")
datacompy_result = evaluate(dataset=ragas_dataset, metrics=[datacompy_metric], column_map={"response": "predicted_csv_output", "reference": "reference_csv_output"})
print("Evaluating LLMSQLEquivalence...")
llm_sql_result = evaluate(dataset=ragas_dataset, metrics=[llm_sql_metric], column_map={"response": "generated_sql", "reference": "gt_query_str", "reference_contexts": "reference_contexts"})

# --- Format and Save Final Results ---
final_df = results_df.copy()

all_score_dfs = {
    'rag': rag_result,
    'datacompy': datacompy_result,
    'llm_sql': llm_sql_result
}
all_score_cols = []

for name, result in all_score_dfs.items():
    scores_df = result.to_pandas()
    new_cols = [col for col in scores_df.columns if col not in final_df.columns]
    if new_cols:
        final_df = final_df.join(scores_df[new_cols])
        all_score_cols.extend(new_cols)

# --- Print Overall Performance Metrics ---
print("\n--- Overall Performance Metrics ---")
print(final_df[all_score_cols].mean(numeric_only=True))
print("----------------------------------")

# Replace blanks and NaN scores with -1
metric_columns_to_keep = [
    'answer_relevancy', 
    'faithfulness', 
    'context_precision', 
    'context_recall', 
    'data_compare_score(mode=rows)', 
    'llm_sql_equivalence_with_reference'
]

# Define the base text columns to keep
text_columns_to_keep = [
    "question", "ground_truth", "gt_query_str", "answer", "generated_sql"
]

# Combine the lists and filter for only columns that actually exist in your DataFrame.
# This prevents errors if a metric failed to run and add its column.
columns_to_keep = text_columns_to_keep + [col for col in metric_columns_to_keep if col in final_df.columns]

# Create the clean DataFrame with ONLY the desired columns
final_df_clean = final_df[columns_to_keep].copy()

# Rename columns for better readability in the final CSV file
final_df_clean.rename(columns={"ground_truth": "gt_answer", "gt_query_str": "gt_query", "answer": "generated_answer"}, inplace=True)

# Save the final, clean DataFrame
final_df_clean.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\n✅ Clean benchmark results saved to {OUTPUT_FILE_PATH}")

Loading benchmark data from /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/sql_benchmark.csv...
Loaded 3 question-answer pairs for evaluation.
--- Running pipeline and collecting data for evaluation ---


  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            | 0/3 [00:00<?, ?it/s]S

Evaluating RAG metrics...


Evaluating:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               | 0/12 [00:00<?, ?it/s]E

Evaluating DataCompyScore...


Evaluating:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                | 0/3 [00:00<?, ?it/s]/

Evaluating LLMSQLEquivalence...


Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:07<00:00,  2.54s/it]



--- Overall Performance Metrics ---
answer_relevancy                           NaN
faithfulness                          0.000000
context_precision                     0.333333
context_recall                        0.000000
data_compare_score(mode=rows)         1.000000
llm_sql_equivalence_with_reference    0.333333
dtype: float64
----------------------------------

✅ Clean benchmark results saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/base_sql_benchmark_results.csv
